# Final Analysis Notebook

This notebook compares **different RAG chunking methods** using **saved evaluation result JSON files** from the `evaluation` folder. 

So please note that it does not rebuild vector stores, call OpenAI for answer generation, or duplicate pipeline logic. All analysis is based on previously saved evaluation outputs rather than real-time generation.


## Metric Definitions

For evaluation, we use the Ragas framework. The selected metrics are summarized below. More details are available in [Ragas available metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/).

### 1. **Context Precision**
- **Focus on _retrieved_ context**: how many of the retrieved context chunks are actually relevant to answering a question.
- **Precision@k**: measures how precise the context is at position _k_.
- **With reference**: compares the retrieved context with the **reference** (aka gold answer), then determines how relevant or helpful that context is in supporting the reference answer.

### 2. **Context Recall**
- **Focus on _retrieved_ context**: how many parts of the gold answer (**reference**) can be found or supported in the retrieved context.
- **Output**:
  - High recall: Good - the retrieved context covers most or all relevant information.
  - Low recall: Bad - the retrieved context misses many relevant pieces.

### 3. **Response Relevancy**
- **Focus on _response_**: how relevant a generated response is to the original **user input**.
- **Output**:
  - Higher score: Good - the response closely matches the intent and content of the user's question.
  - Lower score: Bad - the response may be off-topic, incomplete, or include unnecessary information.

### 4. **Faithfulness**
- **Focus on _response_**: whether the generated response is supported by the **retrieved context**.
- **Output**:
  - `1.0`: Good - fully faithful; all claims are supported by the context.
  - `0.0`: Bad - completely unfaithful; no claim can be verified from the context.


## Environment Setup

In [72]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


pd.set_option("display.max_colwidth", None)


In [73]:
from rag_chunking.config import load_yaml, resolve_project_path
from rag_chunking.data_io import load_json
from rag_chunking.evaluation.analysis import (
    combine_metric_frames,
    confidence_interval,
    load_eval_metrics_all_methods,
    question_type_summary,
    split_question_type_summary,
    summarize_metrics,
)
from rag_chunking.evaluation.ragas_metrics import METRICS

sns.set_theme(style="whitegrid")

## Load Evaluation Files

The file list comes from `configs/analysis.yaml`.


In [74]:
def load_files(in_analysis_config):
    method_files = {
        method: resolve_project_path(path)
        for method, path in in_analysis_config.get("method_files", {}).items()
    }

    missing_files = [path for path in method_files.values() if not Path(path).exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing evaluated result files: {missing_files}")

    display(pd.DataFrame(
        [{"Method": method, "File": Path(path).name} for method, path in method_files.items()]
    ))

    eval_result = load_eval_metrics_all_methods(method_files)
    records_by_method = {method: load_json(path) for method, path in method_files.items()}

    return eval_result, records_by_method, method_files


In [75]:
def check_unique_questions(in_records_by_method):
    all_questions = []
    for records in in_records_by_method.values():
        all_questions.extend(
            record.get("input_question") for record in records if record.get("input_question")
        )

    unique_questions = list(dict.fromkeys(all_questions))
    if not unique_questions:
        raise ValueError("No questions found in loaded evaluation files.")

    return unique_questions


In [76]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "analysis.yaml"
analysis_config = load_yaml(CONFIG_PATH)

eval_result, records_by_method, method_files = load_files(analysis_config)

# Check 1: get the number of questions in each file.
display(pd.DataFrame(
    [{"Method": method, "Questions": len(records)} for method, records in records_by_method.items()]
))

# Check 2: get unique questions across all provided files.
unique_questions = check_unique_questions(records_by_method)
print(f"The files contain {len(unique_questions)} unique questions")


,Method,File
0,Baseline,eval_2025-04-05_gpt-4o_run_results_baseline.json
1,Structured_Baseline,eval_2025-04-05_gpt-4o_run_results_baseline_L1.json
2,Sentence_Window,eval_2025-04-05_run_results_sw_gpt4o_2025-04-05.json
3,Proposition,eval_2025-04-06_run_results_proposition_gpt4o.json


,Method,Questions
0,Baseline,30
1,Structured_Baseline,30
2,Sentence_Window,30
3,Proposition,30


The files contain 30 unique questions


### Single Test

Use this section to inspect **one question** across all methods.

Change `SELECTED_QUESTION` to any question text from the evaluated result files in the `evaluation` folder. Leave it as `None` to use the first shared question.


In [77]:
def get_single_test_result(in_question, in_records_by_method):
    single_test_rows = []
    single_test_contexts = {}

    for method, records in in_records_by_method.items():
        match = next(
            (record for record in records if record.get("input_question") == in_question),
            None,
        )
        if match is None:
            single_test_rows.append(
                {
                    "Method": method,
                    "Found": False,
                    "Response": None,
                    "Retrieved Context Count": 0,
                    **{metric: None for metric in METRICS},
                }
            )
            continue

        evaluation = match.get("Evaluation", {})
        contexts = match.get("retrieved_contexts") or []
        single_test_contexts[method] = contexts
        single_test_rows.append(
            {
                "Method": method,
                "Found": True,
                "Response": match.get("response"),
                "Retrieved Context Count": len(contexts),
                **{metric: evaluation.get(metric) for metric in METRICS},
            }
        )

    single_test_df = pd.DataFrame(single_test_rows)

    return single_test_df


In [83]:
# Set to None to use the first question available in the loaded result files.
SELECTED_QUESTION = "who sings in the eye of the storm"

selected_question = SELECTED_QUESTION or unique_questions[0]
print(f"Evaluating question: {selected_question}")

single_test_df = get_single_test_result(selected_question, records_by_method)
single_test_df


Evaluating question: who sings in the eye of the storm


,Method,Found,Response,Retrieved Context Count,Context_Precision,Context_Recall,Response_Relevancy,Faithfulness
0,Baseline,True,"Ryan Stevenson sings ""Eye of the Storm.""",10,0.8042,0.3333,0.9289,1.0
1,Structured_Baseline,True,"Ryan Stevenson sings ""Eye of the Storm.""",2,1.0000,1.0000,0.9289,1.0
2,Sentence_Window,True,"The song ""Eye of the Storm"" is sung by Christian musician Ryan Stevenson.",10,0.7857,0.6667,0.9211,1.0
3,Proposition,True,Lead vocal by Billy Hoggs.,10,0.7885,0.6667,0.7033,0.0


## Comprehensive Analysis

This is the final evaluation analysis section. It loads evaluated files and runs analysis on **all questions**.


#### Mean&STD


In [79]:
summary_table = summarize_metrics(eval_result)
summary_table


,Context_Precision,Context_Recall,Response_Relevancy,Faithfulness
Method,,,,
Baseline,0.501 (+/-0.399),0.507 (+/-0.352),0.581 (+/-0.425),0.647 (+/-0.471)
Structured_Baseline,0.817 (+/-0.334),0.661 (+/-0.426),0.737 (+/-0.346),0.739 (+/-0.426)
Sentence_Window,0.839 (+/-0.287),0.784 (+/-0.259),0.920 (+/-0.056),0.612 (+/-0.439)
Proposition,0.685 (+/-0.301),0.598 (+/-0.272),0.671 (+/-0.352),0.592 (+/-0.439)


#### Distribution


In [80]:
def plot_violin(metric_to_plot, in_df):
    # Plot a violin plot for each metric.
    for metric in metric_to_plot:
        plt.figure(figsize=(7, 4))

        # Base violin plot.
        ax = sns.violinplot(
            data=in_df,
            x="Method",
            y=metric,
            hue="Method",
            palette="Set2",
            inner="quartile",  # shows median and IQR
            cut=0,
            legend=False,
        )

        # Overlay mean as a black dot.
        for i, system in enumerate(in_df["Method"].unique()):
            mean_val = in_df[in_df["Method"] == system][metric].mean()
            plt.plot(i, mean_val, marker="o", markersize=7, color="black", label="Mean" if i == 0 else "")

        metric_title = metric.replace("_", " ")
        bold_part = metric_title
        normal_part = "(30 Testing Samples)"

        # Plot title manually using `ax.text`.
        ax.text(
            0.5,
            1.05,  # Position above the plot.
            f"{bold_part}",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
        )

        plt.ylabel("Score")
        plt.ylim(0, 1)
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
# Combine all system DataFrames into one long-form DataFrame.
combined_df = pd.concat([
    df.assign(Method=name) for name, df in eval_result.items()
])

# List of metrics to plot.
metrics = metrics = ["Context_Precision", "Context_Recall", "Response_Relevancy", "Faithfulness"]

plot_violin(metrics, combined_df)


#### Question Type Analysis

Original notebook comment retained: `#### Question Type Analysis`, `##### Question lists`, and `##### Run analysis`.

Evaluate different question types, such as fact-based questions and broad questions.


In [82]:
fact_based_questions = analysis_config.get("fact_based_questions") or []
broad_questions = analysis_config.get("broad_questions") or []

if fact_based_questions or broad_questions:
    qtype_summary = question_type_summary(method_files, fact_based_questions, broad_questions)

    fact_based_summary, broad_summary = split_question_type_summary(qtype_summary)

    print("=== Fact-Based Questions Summary ===")
    display(fact_based_summary)

    print("=== Broad Questions Summary ===")
    display(broad_summary)
else:
    print("No question type lists configured. Add questions to configs/analysis.yaml to run this section.")


=== Fact-Based Questions Summary ===


,Method,Context_Precision,Context_Recall,Response_Relevancy,Faithfulness
0,Baseline,0.499,0.529,0.584,0.680
1,Structured_Baseline,0.800,0.634,0.695,0.694
2,Sentence_Window,0.819,0.754,0.922,0.581
3,Proposition,0.717,0.616,0.663,0.580


=== Broad Questions Summary ===


,Method,Context_Precision,Context_Recall,Response_Relevancy,Faithfulness
0,Baseline,0.507,0.394,0.566,0.483
1,Structured_Baseline,0.900,0.800,0.948,0.960
2,Sentence_Window,0.938,0.933,0.910,0.770
3,Proposition,0.522,0.503,0.712,0.653
